# 🎬 RotoDraft Suite — 1-Click Google Colab Cloud Studio
### Automatic Script-to-B-Roll & AI Assets Collector Engine
**The #1 Free, Local-First, Model-Agnostic Open-Source Alternative to Pictory AI, InVideo AI & MoneyPrinter Turbo**

[![GitHub Repository](https://img.shields.io/badge/GitHub-Repository-181717?style=for-the-badge&logo=github)](https://github.com/AliRash3ed/Rotodraft-Suite-AI-Automated-Broll-and-AI-Assets-Collector-Engine)
[![License: MIT](https://img.shields.io/badge/License-MIT-blue.svg?style=for-the-badge)](https://opensource.org/licenses/MIT)
[![Cost: $0 Free](https://img.shields.io/badge/Cost-%240%20Free%20Forever-10b981.svg?style=for-the-badge)](https://github.com/AliRash3ed/Rotodraft-Suite-AI-Automated-Broll-and-AI-Assets-Collector-Engine)

--- 

### ⚡ What This Cloud Studio Does:
1. **Universal Multi-Model AI (BYOK)**: Supports OpenRouter, Google Gemini, Groq, OpenAI, Claude, Cohere, Ollama, DeepSeek & Custom APIs.
2. **9 Parallel Stock Media Vaults**: Pexels, Pixabay, Coverr, Mixkit, Storyblocks, Videvo, Pinterest, Unsplash, Wikimedia.
3. **FFmpeg Hardware Transcoding**: Exact 3.0s trimming, 1080p/4K resolution, 16:9 / 9:16 aspect ratios, and Ken Burns photo motion.
4. **Direct Downloads & NLE Exporters**: 1-Click ZIP archive of all clips, Full Master Video (MP4), Adobe Premiere Pro XML, DaVinci Resolve EDL, and CapCut Draft JSON.

In [ ]:
# @title 🚀 Step 1: Clone Repository & Install Dependencies
# @markdown Run this cell to clone the latest code and install Python & FFmpeg packages.

import os, sys, shutil, subprocess

REPO_URL = "https://github.com/AliRash3ed/Rotodraft-Suite-AI-Automated-Broll-and-AI-Assets-Collector-Engine.git"
DIR_NAME = "rotodraft_suite"

print("📥 [1/4] Cloning RotoDraft Suite repository...")
if os.path.exists(DIR_NAME):
    shutil.rmtree(DIR_NAME)
!git clone {REPO_URL} {DIR_NAME}
os.chdir(f"/content/{DIR_NAME}")

print("📦 [2/4] Installing Python dependencies...")
!pip install -q --upgrade pip
!pip install -q fastapi uvicorn jinja2 python-dotenv requests aiohttp beautifulsoup4 pydantic nest_asyncio psutil
if os.path.exists("requirements.txt"):
    !pip install -q -r requirements.txt

print("🎬 [3/4] Installing FFmpeg multimedia processor...")
!apt-get update -qq && apt-get install -y -qq ffmpeg

print("🌐 [4/4] Installing Cloudflare Tunnel binary for ultra-reliable public URL...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("\n✅ Environment setup completed successfully!")

In [ ]:
# @title 🌐 Step 2: Start RotoDraft Studio & Launch Public Cloud URL
# @markdown Run this cell to start the server and get your live, passwordless web studio link.

import os, sys, time, subprocess, re, urllib.request
from google.colab import output

# Ensure working directory is correct
if not os.path.exists("app.py") and os.path.exists("/content/rotodraft_suite/app.py"):
    os.chdir("/content/rotodraft_suite")

# Kill any previously running instance
!pkill -f "python app.py" || true
!pkill -f "cloudflared" || true
time.sleep(1)

print("🚀 Starting RotoDraft Studio backend server on port 8001...")
log_file = open("/content/rotodraft_server.log", "w")
server_proc = subprocess.Popen(
    [sys.executable, "app.py"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env={**os.environ, "SERVER_PORT": "8001", "SERVER_HOST": "0.0.0.0"}
)

# Health check: wait until server is actively responding on 8001
ready = False
for attempt in range(25):
    time.sleep(1)
    try:
        resp = urllib.request.urlopen("http://127.0.0.1:8001/", timeout=2)
        if resp.status == 200:
            ready = True
            break
    except Exception:
        if server_proc.poll() is not None:
            print("❌ Server process exited prematurely. Error log:")
            log_file.flush()
            with open("/content/rotodraft_server.log", "r") as f:
                print(f.read())
            raise RuntimeError("Backend server failed to start.")

if not ready:
    print("⚠️ Server took longer than expected to bind. Showing logs:")
    with open("/content/rotodraft_server.log", "r") as f:
        print(f.read())
else:
    print("✅ Backend server is LIVE on http://127.0.0.1:8001!")

# Launch Cloudflare Tunnel
print("\n🌍 Launching secure Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

cloud_url = None
start_t = time.time()
while time.time() - start_t < 20:
    line = tunnel_proc.stdout.readline()
    if not line:
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        cloud_url = match.group(0)
        break

print("=" * 75)
if cloud_url:
    print(f"🎉 ROTODRAFT SUITE IS READY!")
    print(f"👉 Primary Public URL: {cloud_url}")
    print("=" * 75)
    print("💡 Note: Open the link above in a new browser tab to access the studio!")
else:
    print("⚠️ Cloudflare Tunnel URL not detected in time. Using Colab direct port proxy:")
    output.serve_kernel_port_as_window(8001)
